**Set the GPU runtime before running any cell.**

1. Runtime > Change runtime type.
2. Hardware accelerator: T4 GPU. Click Save.

![Runtime menu, Change runtime type](https://raw.githubusercontent.com/Silent0Wings/ShapeSense/main/assets/ScreenShot/change%20to%20T4_Change%20runtime%20type.png)

![Change runtime type dialog with T4 GPU selected](https://raw.githubusercontent.com/Silent0Wings/ShapeSense/main/assets/ScreenShot/T4%20GPU.png)


**Disclaimer: this notebook has not been run end to end.** It has no saved outputs.

Known issues, found by a read-only audit on 2026-11-20:

- Run it on a GPU runtime (Runtime > Change runtime type > T4 GPU). The cell that checks for CUDA stops the notebook on a CPU runtime.
- Run the cells in order from the top. Stages 2 to 6 depend on earlier outputs.
- Cells that use `json` do not import it themselves (cells 26 and 30). Run the import cell first.
- The raw data comes from the GitHub release `data_set` in the data cell. The repo is private, so that cell needs a GitHub token stored as the Colab secret `GITHUB_TOKEN`. The cell is untested.
- The install step uses the default PyPI torch build, not the CUDA 13.0 build used for the recorded results, so numbers can differ.
- Cell 18 overwrites `results/improvement_table.json` and logs a test-split read in `artifacts/test_access.log`.
- The sealed fresh holdouts are regenerated by project scripts and may not match the recorded ones exactly.

Recorded results are in `results/`. This notebook is a copy of the pipeline steps, not a verified re-run.

# ShapeSense: final pipeline

Runs top to bottom. One section per stage. Set `DATA_DIR` to the folder with the raw images (`data_set/`), which is not in the repository.

Colab link: <https://colab.research.google.com/github/Silent0Wings/ShapeSense/blob/main/notebooks/shapesense_pipeline.ipynb>

## 0. Runtime check (needs a CUDA GPU)

**Why this cell requires a GPU.** The pipeline is set to `device: cuda` in `configs/pipeline.yaml`, and it stops when no GPU is visible rather than switching to CPU. A CPU run would not match the recorded GPU results. The stages load saved models with `map_location="cpu"`, so a model trained on CUDA loads on any machine. If you only want to run a finished model for inference, you can use `device = torch.device("cuda" if torch.cuda.is_available() else "cpu")` with `torch.load(path, map_location=device)` in a separate cell.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Switch to a GPU runtime: Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))

## 1. Get the code and install pins

In [ ]:
!git clone https://github.com/Silent0Wings/ShapeSense.git
%cd ShapeSense
!pip install -q -r requirements.txt

## 2. Raw data

Mount Drive and link the raw images. The raw data for the four sources used is about 730 MB and is listed in `data/README.md`. The 35 MB figure is the generated unified dataset that Stage 1 writes, not the raw input.

In [ ]:
# Download the source data from the GitHub release and unpack it into the clone.
# Public repo: no token needed. Private repo: store a GitHub token as the Colab secret GITHUB_TOKEN.
# Untested: check the output of the cell.
import json, os, shutil, urllib.request, zipfile

try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None

REPO = 'Silent0Wings/ShapeSense'

def gh(url, accept='application/vnd.github+json'):
    headers = {'Accept': accept}
    if TOKEN:
        headers['Authorization'] = f'Bearer {TOKEN}'
    return urllib.request.urlopen(urllib.request.Request(url, headers=headers))

if not os.path.isdir('data_set/quickdraw'):
    if TOKEN:
        release = json.load(gh(f'https://api.github.com/repos/{REPO}/releases/tags/data_set'))
        url = next(a['url'] for a in release['assets'] if a['name'] == 'shapesense_raw_sources_v1.zip')
        accept = 'application/octet-stream'
    else:
        url = f'https://github.com/{REPO}/releases/download/data_set/shapesense_raw_sources_v1.zip'
        accept = '*/*'
    with gh(url, accept=accept) as r, open('/content/data_set.zip', 'wb') as f:
        shutil.copyfileobj(r, f)
    zipfile.ZipFile('/content/data_set.zip').extractall('.')

print('data_set folders:', sorted(os.listdir('data_set')))


## 3. Guards and contract tests

In [ ]:
!python run_pipeline.py --check

## 4. Stage 1: data freeze and splits

Builds the unified dataset, applies the frozen cleaning rule (`exclude_source_labels` in `configs/pipeline.yaml`: Quick, Draw! diamonds), removes exact duplicates and blank images, and writes the group-aware splits. The raw baseline in section 8 is rebuilt from the raw files by replaying this step.

In [ ]:
!python run_pipeline.py --stages 1

## 5. Stage 2: augmentation (transformation)

Class-safe augmentation. F1 after this step: see section 8.

In [ ]:
!python run_pipeline.py --stages 2

## 6. Stage 3: synthetic generator

In [ ]:
!python run_pipeline.py --stages 3

## 7. Stage 4 and 5: training and test evaluation

Training is per arm and per seed. Test is read once per run (see `artifacts/test_access.log`).

In [ ]:
!python run_pipeline.py --stages 4 5

## 8. Baseline and improvement after each data preparation step

Every row trains the same CNN (same settings, 49 steps x 60 epochs, seeds 0-4) and is scored by **macro F1 on the same 2,776 test images**. Only the data preparation changes. `scripts/improvement_table.py` documents each row.

1. `replay` rebuilds every image from its raw file, only decoded and resized to 64x64.
2. `train` trains the baseline (raw, uncleaned) and the cleaning-only model.
3. `fs` picks the input resolution on validation (feature selection).
4. `evaluate` reads the test split once, through Stage 5's logged reader.

In [ ]:
!python -m scripts.improvement_table replay
!python -m scripts.improvement_table train
!python -m scripts.improvement_table fs
!python -m scripts.improvement_table evaluate

import json
table = json.load(open('results/improvement_table.json'))
rows = table['rows']
def show(i):
    r = rows[i]
    print(r['step'])
    print(f"  test macro F1 {r['mean']:.2f} +/- {r['std']:.2f} (5 seeds), delta vs baseline {r['delta_vs_baseline']:+.2f}")


### 8.1 Baseline: the collected data before any preparation

In [ ]:
show(0)

### 8.2 After cleaning

In [ ]:
show(1)

### 8.3 After transformation (standardization, then augmentation + synthetic shapes)

In [ ]:
show(2); show(3)

### 8.4 After feature selection

In [ ]:
import json
show(4)
print(json.dumps(table['feature_selection'], indent=2))

## 9. Stage 6: report

In [ ]:
!python run_pipeline.py --stages 6

## 10. Final model vs. direct LLM prediction on the same test images

200 test images (20 per class, seed 0, `results/direct_llm_prediction/sample_ids.csv`) were shown blind to Claude Opus 5.5 and Gemini 4 Neon. The trained CNN (A2_0) is scored on the same images with the same metric. The baseline model's score on these 200 images is printed too.

In [ ]:
import json
import csv, numpy as np
from sklearn.metrics import f1_score
def macro(path):
    rows = list(csv.DictReader(open(path)))
    return 100 * f1_score([r['true_label'] for r in rows], [r['predicted_label'] for r in rows], average='macro')
for name in ['trained_cnn', 'claude_opus', 'gemini_4_neon']:
    print(f"{name:14s} macro F1 {macro(f'results/direct_llm_prediction/{name}_predictions.csv'):.2f}")
sub = json.load(open('results/improvement_table.json'))['direct_llm_subset']
print(f"baseline (raw) macro F1 {sub['baseline_0']:.2f}; final A2_0 recomputed {sub['final_A2_0']:.2f}")
